# Prophet Forecasting - Basics

This notebook walks through time-series forecasting step by step using Facebook Prophet.

## What you'll learn
1. What time-series data looks like
2. How Prophet decomposes a signal into trend + seasonality
3. How to fit a model and generate forecasts
4. How to evaluate accuracy (MAE, RMSE)
5. How to run 'what if' scenario comparisons

## Step 1: Install & Import

In [ ]:
# Run this cell once to install dependencies
# !pip install prophet pandas matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

print('All imports successful!')

## Step 2: Load & Inspect the Data

In [ ]:
df = pd.read_csv('../data/sample_data.csv')
df['ds'] = pd.to_datetime(df['ds'])

print(f'Shape: {df.shape}')
print(df.head(10))

# Plot the raw data
plt.figure(figsize=(12, 4))
plt.plot(df['ds'], df['y'], color='steelblue')
plt.title('Raw Time Series')
plt.xlabel('Date')
plt.ylabel('Value')
plt.tight_layout()
plt.show()

## Step 3: Train / Test Split

In [ ]:
# Use 80% for training, 20% as held-out test set
split_idx = int(len(df) * 0.8)
train = df.iloc[:split_idx].copy()
test  = df.iloc[split_idx:].copy()

print(f'Train rows: {len(train)}')
print(f'Test rows:  {len(test)}')

## Step 4: Fit the Prophet Model

In [ ]:
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    interval_width=0.95
)

model.fit(train)
print('Model trained!')

## Step 5: Forecast & Visualise

In [ ]:
# Forecast 90 days into the future
future = model.make_future_dataframe(periods=90)
forecast = model.predict(future)

# Prophet's built-in plot
fig1 = model.plot(forecast)
fig1.suptitle('Prophet Forecast (black dots = actual, blue = prediction, shaded = uncertainty)', fontsize=10)
plt.show()

# Components: trend + weekly + yearly seasonality
fig2 = model.plot_components(forecast)
plt.show()

## Step 6: Evaluate on Test Set

In [ ]:
# Merge test actuals with forecast predictions
merged = test.merge(forecast[['ds', 'yhat']], on='ds', how='inner')

mae  = mean_absolute_error(merged['y'], merged['yhat'])
rmse = np.sqrt(mean_squared_error(merged['y'], merged['yhat']))

print(f'MAE  = {mae:.2f}  (average error per prediction)')
print(f'RMSE = {rmse:.2f}  (penalises large errors more)')

# NOTE: Lower is better for both metrics.

## Step 7: What-If Scenario Comparison

Prophet lets you add custom regressors (external factors) and manually change future values.
Below is a simplified example: compare a baseline forecast vs an optimistic scenario (values +10%).

In [ ]:
# Baseline: already computed above as 'forecast'
baseline = forecast[['ds', 'yhat']].copy()
baseline.columns = ['ds', 'baseline']

# Optimistic scenario: shift the forecast up by 10%
optimistic = forecast[['ds', 'yhat']].copy()
optimistic['yhat'] = optimistic['yhat'] * 1.10
optimistic.columns = ['ds', 'optimistic']

# Pessimistic scenario: shift down by 10%
pessimistic = forecast[['ds', 'yhat']].copy()
pessimistic['yhat'] = pessimistic['yhat'] * 0.90
pessimistic.columns = ['ds', 'pessimistic']

# Combine and plot future window only
scenarios = baseline.merge(optimistic, on='ds').merge(pessimistic, on='ds')
future_mask = scenarios['ds'] > df['ds'].max()
future_scenarios = scenarios[future_mask]

plt.figure(figsize=(12, 5))
plt.plot(future_scenarios['ds'], future_scenarios['baseline'],    label='Baseline', color='steelblue')
plt.plot(future_scenarios['ds'], future_scenarios['optimistic'],  label='Optimistic (+10%)', color='green', linestyle='--')
plt.plot(future_scenarios['ds'], future_scenarios['pessimistic'], label='Pessimistic (-10%)', color='red',   linestyle='--')
plt.title('Scenario Comparison: Next 90 Days')
plt.xlabel('Date')
plt.ylabel('Forecasted Value')
plt.legend()
plt.tight_layout()
plt.show()

## Next Steps

- Replace `sample_data.csv` with a real dataset (Singapore weather, stock prices, etc.)
- Try adding **holidays** using `model.add_country_holidays(country_name='SG')`
- Add **external regressors** (e.g. temperature affects sales)
- Run the clean script version: `python src/forecast.py --data data/your_data.csv --periods 90`
- Build a Streamlit dashboard for interactive forecasting